# Model performance

Leave-one-country-out performance for all ten fitted model variants, the check that each reproduces
its shipped pickle, and the published performance figure regenerated in the project palette.

The computation lives in `src/model_performance.py`. Read `doc/modelling.md` first if the variant
names are unfamiliar — three of them do not mean what they say (D37).

In [ ]:
import _bootstrap  # noqa: F401  — puts src/ on sys.path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import model_performance as mp
import params
import plotting as plot

STAGE = 'model_performance'
CFG = params.FINAL_MODEL

OUTCOME_LABEL = {'ggi': 'GGI', 'wom': 'Women level', 'men': 'Men level'}
VARIANT_LABEL = {'online_with_CIS': 'Online', 'offline_with_CIS': 'Offline',
                 'combined_with_CIS': 'Combined'}

print('production variant:', CFG['model_type'])
print('figure variants   :', params.FIGURE_VARIANTS)

## 1. What each variant actually is

All ten read the same file and differ in two things: whether the ITU rows are kept, and the
regressor list. `_with_CIS` means **keep the ITU rows** — it does not select CIS countries.

In [ ]:
pd.DataFrame([
    {'variant': v, 'keep_itu': c['keep_itu'], 'align': c.get('align', False),
     'spec': ' + '.join(c['spec']),
     'production': v == CFG['model_type']}
    for v, c in params.MODEL_VARIANTS.items()
])

## 2. Run the audit

Refits every variant, computes LOCO, and compares against every shipped pickle. Takes a few
minutes; the tables are date-stamped so a rerun never overwrites an earlier vintage.

In [ ]:
perf, checks = mp.evaluate(list(params.MODEL_VARIANTS))

print(f"{len(checks)} pickle comparisons | samples match: {bool(checks['n_matches'].all())} "
      f"| specs match: {bool(checks['spec_matches'].all())} "
      f"| max |coefficient difference|: {checks['max_abs_coef_diff'].max():.2e}")
perf[perf['is_production']][['indicator', 'outcome', 'n', 'r2', 'adj_r2',
                             'loco_r2', 'loco_mae']].round(3)

## 3. The published performance figure

Leave-one-country-out R², online / offline / combined, on the production sample (108 internet,
99 mobile). Redrawn in the project palette, which is rebuilt for colour-vision accessibility
(D23) — the original used a different set.

The bar height is what matters, so the grid sits on the value axis only and each bar carries its
value; `n` is constant within a panel and is stated in the subtitle instead.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharey=True)

series = params.FIGURE_VARIANTS
colours = dict(zip(series, plot.PALETTE))
outcomes = ['ggi', 'wom', 'men']

for ax, indicator in zip(axes, CFG['indicators']):
    block = perf[(perf['indicator'] == indicator) & (perf['variant'].isin(series))]
    values = {v: [float(block[(block['variant'] == v) & (block['outcome'] == o)]['loco_r2'].squeeze())
                  for o in outcomes] for v in series}
    plot.grouped_bars(ax, [OUTCOME_LABEL[o] for o in outcomes], series, values,
                      colours=colours, fmt='{:.2f}')
    n = int(block['n'].iloc[0])
    plot.tidy(ax, title=f'{indicator.capitalize()}-related outcomes  ·  n = {n}', grid='y')
    ax.set_ylim(0, 1.0)

axes[0].set_ylabel('Leave-one-country-out  R²', fontsize=plot.STYLE['tick_fs'], color=plot.MUTED)
plot.legend(fig, [plot.swatch(colours[v], VARIANT_LABEL[v]) for v in series], y=-0.06)
plot.suptitle(fig, 'Adult internet and mobile outcomes — model performance')
fig.tight_layout(rect=(0, 0, 1, 0.93))
plot.save(fig, STAGE, '03_01_loco_r2_by_variant')
plt.show()

### Against the originally published values

The figure this reproduces printed: internet 0.74 / 0.72 / 0.79 (GGI), 0.73 / 0.86 / 0.88 (women),
0.68 / 0.79 / 0.84 (men); mobile 0.64 / 0.61 / 0.68, 0.64 / 0.62 / 0.75, 0.38 / 0.40 / 0.56.

In [ ]:
PUBLISHED = {('internet','ggi'):(0.74,0.72,0.79), ('internet','wom'):(0.73,0.86,0.88),
             ('internet','men'):(0.68,0.79,0.84), ('mobile','ggi'):(0.64,0.61,0.68),
             ('mobile','wom'):(0.64,0.62,0.75), ('mobile','men'):(0.38,0.40,0.56)}

rows = []
for (ind, out), published in PUBLISHED.items():
    got = tuple(round(float(perf[(perf['indicator'] == ind) & (perf['variant'] == v)
                                 & (perf['outcome'] == out)]['loco_r2'].squeeze()), 3)
                for v in params.FIGURE_VARIANTS)
    rows.append({'indicator': ind, 'outcome': out,
                 'published': ' / '.join(f'{v:.2f}' for v in published),
                 'recomputed': ' / '.join(f'{v:.3f}' for v in got),
                 'max_abs_diff': round(max(abs(a - b) for a, b in zip(published, got)), 3)})
check = pd.DataFrame(rows)
print(f"largest disagreement with the published figure: {check['max_abs_diff'].max():.3f}")
check

## 4. Every variant, for reference

Including the suffix-less `online` / `offline` / `combined`, which are fitted on the **71 / 75**
sample and give materially lower numbers. Reading the published figure off those is the natural
mistake (D37).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True)

order = [v for v in params.MODEL_VARIANTS if not v.endswith('_align')]
for ax, indicator in zip(axes, CFG['indicators']):
    block = perf[(perf['indicator'] == indicator) & (perf['variant'].isin(order))]
    y = np.arange(len(order))
    for k, outcome in enumerate(outcomes):
        vals = [float(block[(block['variant'] == v) & (block['outcome'] == outcome)]['loco_r2'].squeeze())
                for v in order]
        ax.scatter(vals, y + (k - 1) * 0.22, s=70, color=plot.PALETTE[k],
                   edgecolor='white', linewidth=0.8, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{v}  (n={int(block[block['variant'] == v]['n'].iloc[0])})" for v in order])
    ax.set_ylim(len(order) - 0.5, -0.5)
    plot.tidy(ax, title=indicator.capitalize(), xlabel='Leave-one-country-out  R²', grid='x')
    ax.set_xlim(0, 1)

plot.legend(fig, [plot.swatch(plot.PALETTE[k], OUTCOME_LABEL[o], marker='o')
                  for k, o in enumerate(outcomes)], y=-0.05)
plot.suptitle(fig, 'All model variants — the sample is the difference, not the specification')
fig.tight_layout(rect=(0, 0, 1, 0.94))
plot.save(fig, STAGE, '03_01_loco_r2_all_variants')
plt.show()

## 5. Write the tables

In [ ]:
mp.main(list(params.MODEL_VARIANTS))

## Notes

- LOCO R² is pooled (`1 − RSS/TSS`) over held-out predictions, not the mean of per-fold R²: with
  one to three rows per country a per-fold R² is undefined.
- `doc/modelling.md` is the written account — the spec, every filter, the ten variants and the
  three misleading names.
- The survey validation of these predictions is `03_02_unseen_survey_validation.ipynb`.